In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as f

In [6]:
class CLIP(nn.Module):
    def __init__(
        self,
        text_dim: int,
        img_dim: int,
        emb_dim: int,
        temp: torch.float,
        batch_size: int
    ):
        super().__init__()
        self.Wi = nn.Linear(img_dim, emb_dim, bias=False)
        self.Wt = nn.Linear(text_dim, emb_dim, bias=False)
        self.register_buffer(
            "labels", torch.arange(batch_size)
        )
        self.temp = temp

    def forward(self, text_enc, img_enc):
        text = self.Wt(text_enc) # b, emb_dim
        img = self.Wi(img_enc) # b,emb_dim
        text = f.normalize(text, dim=-1)
        img = f.normalize(img, dim=-1)
        logits = torch.einsum("ie,je-> ij", [img, text]) / self.temp # b, b
        loss_i = f.cross_entropy(logits, self.labels)
        loss_t = f.cross_entropy(logits.T, self.labels)
        loss = (loss_i + loss_t) / 2
        return loss

In [ ]:
batch_size = 4
text_dim = 8
img_dim = 16
emb_dim = 32

model = CLIP(
    text_dim=text_dim,
    img_dim=img_dim,
    emb_dim=emb_dim,
    temp=0.07,
    batch_size=batch_size,
)

text_enc = torch.randn(batch_size, text_dim)
img_enc = torch.randn(batch_size, img_dim)

loss = model(text_enc, img_enc)
loss.backward()

assert loss.ndim == 0
assert torch.isfinite(loss)
print(f"Loss: {loss.item():.4f}")

torch.Size([])
Loss: 3.7604
torch.Size([])
